# **Extreme Water Levels: GEV Analysis for all 12 months of the year**

Last updated on: 09/22/2026

## Overview
NOAA's Center for Operational Oceanographic Products and Services (CO-OPS) is the nation's source for [coastal inundation data](https://tidesandcurrents.noaa.gov) through its network of long-term water level gauges. In this notebook, we will walk through the steps for completing a 12-month (seasonal) [GEV extreme value analysis](https://repository.library.noaa.gov/view/noaa/14420).

This notebook is designed to explore a single station. Users can choose their station of interest here: [Trends and Extremes Map](https://tidesandcurrents.noaa.gov/trends-and-extremes/index.html). For this example we will use [Sewells Point, VA (8638610)](https://tidesandcurrents.noaa.gov/trends-and-extremes/main-product.html?mapTab=station&tab=extremes&station-id=8638610&plot=extreme-water-levels).

## Prerequisites and running the Notebook
This notebook uses Python programming and users may run it either in **Google Colab** or **Jupyter Notebook**.

* **Jupyter Notebook:** Recommended if you prefer to work offline and manage libraries and dependencies personally.
* **Google Colab:** The best option for cloud-based instant access, seamless cooperation, and if you have a good internet connection.

This notebook is designed so that people with limited familiarity with Python can run it.

### Using Google Colab

There are two ways to open this notebook in Colab.

**Method 1: Open directly from GitHub (Recommended)**

1.  Open [Google Colab](https://colab.research.google.com/).
2.  In the menu bar, go to **File > Open Notebook**.
3.  In the popup window, click on the **GitHub** tab.
4.  Paste the repository URL into the search bar and press Enter:
    `https://github.com/NOAA-CO-OPS/Coastal_Hazards_Example_Notebooks`
5.  Click on `CO-OPS_12month_GEV_Extreme_Value_Analysis_Notebook/CO-OPS_12month_GEV_Extreme_Value_Analysis_Notebook.ipynb` to open it.

**Method 2: Download and Upload (All Files)**

This method downloads the entire Coastal_Hazards_Example_Notebooks repository.

1.  Go to the [main GitHub repository](https://github.com/NOAA-CO-OPS/Coastal_Hazards_Example_Notebooks).
2.  Click the green **Code** button, then select **Download ZIP**.
3.  Extract the ZIP file on your computer. This folder contains the notebook (`.ipynb`).
4.  Open [Google Colab](https://colab.research.google.com/).
5.  Go to **File > Upload notebook...** and select the `CO-OPS_12month_GEV_Extreme_Value_Analysis_Notebook.ipynb` file from the folder you just extracted.

### Using Jupyter Notebook

This method downloads the entire Coastal_Hazards_Example_Notebooks repository.

1.  Go to the [main GitHub repository](https://github.com/NOAA-CO-OPS/Coastal_Hazards_Example_Notebooks).
2.  Click the green **Code** button, then select **Download ZIP**.
3.  Extract the ZIP file on your computer.
4.  Start Jupyter Notebook on your local machine and navigate to the extracted folder (`.../CO-OPS_12month_GEV_Extreme_Value_Analysis_Notebook/`) to open `CO-OPS_12month_GEV_Extreme_Value_Analysis_Notebook.ipynb`.

### ▶️ Running the Notebook

Once you have the notebook open:

1.  [Explore different station/s](https://tidesandcurrents.noaa.gov/est/) and decide which you want to analyze.
2.  In **Step 3** of the notebook, change the **station variable** and **unit of interest**.
3.  In the top menu, click **Runtime** and choose **Run All**.
4.  When the 'Choose Files' box in **Step 2** becomes active, click on it.
5.  You will see a green check mark to the left of each cell as it is successfully executed.

Users should be prepared to commit 30 minutes to familiarize themselves with the notebook, upload data, run code and examine results.

## Learning Outcomes
In this notebook users will be able to:
  1.  Prepare a monthly maxima or minima time series for a GEV analysis by first detrending the series.
  2.  Run the GEV analysis on monthly (Jan, Feb, Mar...) detrended data
  3.  Examine and plot the results as they should appear when a station is published to the website.

## Data Source and Documentations
Input and output documentation can be found in the [CO-OPS Data API](https://api.tidesandcurrents.noaa.gov/api/prod/), [CO-OPS Derived Product API v0.1](https://api.tidesandcurrents.noaa.gov/dpapi/prod/) and the [CO-OPS Metadata API](https://api.tidesandcurrents.noaa.gov/mdapi/prod/).

Standard templates are available within the [CO-OPS API URL Builder](https://tidesandcurrents.noaa.gov/api-helper/url-generator.html), but this notebook is designed to give you more flexiblity through building our own API template to generate queries from our Data API, MetaData API, and Derived Product API. All data used for this notebook come from the CO-OPS API


## Software
This tutorial uses the following Python packages:
- datetime
- requests
- time
- pandas
- numpy
- matplotlib
- math
- plotly
- warnings
- noaa-coops


This tutorial also uses rpy2.ipython and rpy2.robjects.packages to be able to load the R package "extRemes" and use it in a Colab or Jupyter notebook.

## **Step 1**
## Install and Load Packages
Lets check if you have these packages installed. If not, this cell will install them.

In [ ]:
# @title
import subprocess, sys
packages = ["datetime","requests","time","pandas","numpy","matplotlib","warnings","math","plotly","noaa-coops"]
for package in packages:
    try:
        __import__(package)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])

In [ ]:
# @title Importing python packages
import datetime as dt
from datetime import timedelta
import requests
import time as time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import math
from math import exp
import plotly.graph_objects as go
from noaa_coops import Station

#this is to suppress user warnings
warnings.filterwarnings("ignore")

In [ ]:
# @title Load the rpy2 extension for using R in a Python environment. This should take about a minute and a half.
%load_ext rpy2.ipython

import rpy2.robjects.packages as rpackages
utils = rpackages.importr('utils')
utils.chooseCRANmirror(ind=1)

utils.install_packages('extRemes')

(as ‘lib’ is unspecified)








	‘/tmp/RtmpD5oM17/downloaded_packages’



<rpy2.rinterface_lib.sexp.NULLType object at 0x78f6f2503390> [0]

## **Step 2**
## Here we choose the station of interest
Click [here](https://tidesandcurrents.noaa.gov/sltrends/sltrends.html) to see a map of stations to get Station IDs

### The stations with the longest periods of record (greater than 95 years) are:
|Station ID and Name  | Station ID and Name | Station ID and Name|
|:--------------------|:--------------------|:--------------------|
|1612340	Honolulu, HI	| 8574680	Baltimore, MD	| 8771450	Galveston, TX |
|1617760	Hilo, HI	| 8575512	Annapolis, MD	| 9410170	San Diego, CA |
|8410140	Eastport, ME	|	8594900	Washington, DC	| 9410230	La Jolla, CA|
|8418150	Portland, ME	|	8638610	Sewells Point, VA	| 9410660	Los Angeles, CA|
|8443970	Boston, MA	| 8665530	Charleston, SC	| 9414290 San Francisco, CA|
|8452660	Newport, RI	| 8720030	Fernandina Beach, FL | 9439040	Astoria, OR|
|8518750	The Battery, NY	|	8724580	Key West, FL| 9447130	Seattle, WA|
|8534720	Atlantic City, NJ	|	8727520	Cedar Key, FL | 9450460	Ketchikan, AK|
|8557380	Lewes, DE	| 8729840	Pensacola, FL| 9451600	Sitka, AK |

In [ ]:
#input station ID below
station = '8638610'

#options are metric or english
units = 'metric'

#options are high for maxima or low for minima
levelType = 'high'

## **Step 3**
## Next, we pull in our station's relative sea level trend and the monthly extremes data from CO-OPS APIs

In [ ]:
# @title Station Metadata
print(f"Fetching data for Station ID: {station}...\n")

# Initialize the station object (this downloads the metadata)
station_meta = Station(station)

# General Metadata
stationName = station_meta.name
print(f"Station Name: {stationName}")
print(f"Coordinates (Lat, Lon): {station_meta.lat_lon}")

station_info = station+' '+stationName

metadata_dict = station_meta.metadata
established_date = metadata_dict.get('details', {}).get('established', 'Unknown')
print(f"Established Date: {established_date}\n")


# Datum & Epoch Information
# Access the specific 'datums' dictionary within the metadata
datums_meta = metadata_dict.get('datums', {})

# Grab the epoch info, defaulting to '1983-2001' if missing or '0000-0000'
epoch_info = datums_meta.get('epoch', '0000-0000')
if epoch_info == '0000-0000' or not epoch_info:
    epoch_info = '1983-2001'

epic_start = int(epoch_info[0:4])
epic_end = int(epoch_info[5:])
epic_length = epic_end - epic_start
epic_mid = int(epic_start + epic_length / 2)

print(f"Epoch: {epic_start}-{epic_end}; Epoch Mid: {epic_mid}")


# Control vs. Subordinate Station
# .strip() removes any accidental spaces from the API response
ctrl_station = datums_meta.get('ctrlStation', '').strip()

if not ctrl_station:
    # If the control station string is empty, this is a Control Station
    print(f"Hierarchy: Station {station} is a Control Station.")
else:
    # If a control station ID exists, this is a Subordinate Station
    print(f"Hierarchy: Station {station} is a Subordinate Station.")
    print(f"Control Station ID: {ctrl_station}")

    # Extract the comparison dates from the DatumAnalysisPeriod list
    analysis_period_list = datums_meta.get('DatumAnalysisPeriod', ['Dates not available'])
    if analysis_period_list:
        analysis_period = analysis_period_list
        print(f"Dates of Comparison: {analysis_period}")

Fetching data for Station ID: 8638610...

Station Name: Sewells Point
Coordinates (Lat, Lon): {'lat': 36.94278, 'lon': -76.32861}
Established Date: 1927-07-01 00:00:00

Epoch: 1983-2001; Epoch Mid: 1992
Hierarchy: Station 8638610 is a Control Station.


In [ ]:
# @title Datum details
#grab datum info
server = 'https://api.tidesandcurrents.noaa.gov/mdapi/prod/webapi/stations/'

myurl = (server + station + '/datums.json?units='+units)

urlResponse = requests.get(myurl)
content=urlResponse.json()

# Assign data to variables.
epoch_info = content['epoch']
datums_info = pd.DataFrame(content['datums'])

#if we can't find the epoch info, default to NTDE
if epoch_info == '0000-0000':
  print('Datum info not found, defaulting to NTDE')
  if station == '1770000':
    epoch_info = '2010-2018'
  else:
    epoch_info = '1983-2001'
#Special case stations with modified datum
if station in ['9462620','8770570','8771450','8770822','8771510','8772447','9455920','8770613','8771013', '8771341', '8772471']:
  epoch_info = '1997-2001'

epic_start = int(epoch_info[0:4])
epic_end = int(epoch_info[5:])
epic_length = epic_end - epic_start
epic_mid = int((epic_start + epic_length/2))

print('Epic midpoint for '+ station + ' is ' + str(epic_mid))

Epic midpoint for 8638610 is 1992


In [ ]:
# @title Get the monthly extremes
#getting the monthly maxima or minima

# set datum depending on levelType
if levelType == 'high':
  datum = 'mhhw'
else:
  datum = 'mllw'

extremeType = 'monthlies'
server = 'https://api.tidesandcurrents.noaa.gov/dpapi/uat/webapi/product'

myurl = server  + '.json?station=' + station + '&name=extremewaterlevels' + '&extremeType=' + extremeType + '&levelType=' + levelType + '&units=metric' + '&datum=' + datum

# Use "requests" to get the Monthly Extremes Data
urlResponse = requests.get(myurl)
content=urlResponse.json()

# Assign data to variables.
myextremes = content['extremes']

*** Disclaimer: A linear trend across the entire Period of Record (POR) is used for both detrending and later plotting the monthly extremes on CO-OPS's website. For the handful of earthquake impacted stations, we currently only provide the post-earthquake recent trend in our extremes api endpoint. Therefore the calculations for earthquake stations done in this notebook may not match the offical CO-OPS calculations seen on the website.

In [ ]:
# @title Grab the POR trend rate used for detrending and plotting
if units == 'metric':
  mytrendrate = content['trend']/1000
  trend_text = str(round(mytrendrate*1000,2)) + ' mm/year'
else:
  mytrendrate = content['trend']/120
  trend_text = str(round(mytrendrate*120,2)) + ' inches/decade'

trendstartdate = content['trendStartDate']
trendenddate = content['trendEndDate']

## **Step 4**

In [ ]:
# @title Detrending monthly extremes

unit_abbr = 'm' if units == 'metric' else 'ft'

# 1. Formatting the data and creating the datetime column
myextremes = pd.DataFrame(myextremes)
myextremes['date'] = pd.to_datetime(myextremes['year'].astype(str) + '-' + myextremes['month'].astype(str) + '-15')

# 2. Convert start and end variables to pandas datetime objects for safe comparison
start_dt = pd.to_datetime(trendstartdate)
end_dt = pd.to_datetime(trendenddate)

# 3. Filter the dataframe to only include rows within the specified date range
myextremes = myextremes[(myextremes['date'] >= start_dt) & (myextremes['date'] <= end_dt)]

# 4. Calculate relative time and detrend
reference_date = pd.to_datetime(f'{epic_mid}-07-01')
myextremes['time_relative'] = (myextremes['date'] - reference_date).dt.days / 365.25

# For a single trend, the intercept is zero.
myextremes['detrended_value'] = myextremes['value'] - (myextremes['time_relative'] * mytrendrate)

# 5. Plotting the results
fig = go.Figure()

# Add the line plot with a dynamic hover template
fig.add_trace(go.Scatter(
    x=myextremes['date'],
    y=myextremes['detrended_value'],
    mode='lines',
    name='Detrended Value',
    # Breakdown of the f-string below:
    # 1. Date: %{x|%Y-%m-%d} -> Shows date in YYYY-MM-DD format.
    # 2. <br> -> Adds a line break.
    # 3. Value: %{y:.2f} -> Shows Y-value with 2 decimal places.
    # 4. {unit_abbr} -> Inserts your Python variable ('m' or 'ft').
    # 5. <extra></extra> -> Hides the trace name box on the side.
    hovertemplate=f'Date: %{{x|%Y-%m-%d}}<br>Value: %{{y:.2f}} {unit_abbr}<extra></extra>'
))

# Update the layout with a dynamic y-axis title
fig.update_layout(
    title=f'Detrended Monthly Extremes {station} {stationName}',
    xaxis_title='Date',
    yaxis_title=f'Detrended Extreme Water Level ({unit_abbr} relative to {datum})',
    template='plotly_white'
)

# Show the interactive plot
fig.show()

In [ ]:
# @title If levelType = 'low' take the negative of monthly extremes

# warning: each time this cell is run, the data is inverted

# the EVA code from the R extRemes package assumes that the most positive value is the most extreme value

# @title If levelType = 'low' take the negative of monthly extremes

# warning: each time this cell is run, the data is inverted

# the EVA code from the R extRemes package assumes that the most positive value is the most extreme value

if levelType == 'low':
  myextremes['detrended_value'] = myextremes['detrended_value'] * -1

  # Create an empty figure object
  fig = go.Figure()

  # Add the line plot trace with a dynamic hover template
  fig.add_trace(go.Scatter(
      x=myextremes['date'],
      y=myextremes['detrended_value'],
      mode='lines',
      name='Detrended Level',
      # Breakdown of the f-string below:
      # 1. Date: %{{x|%Y-%m-%d}} -> Shows date in YYYY-MM-DD format.
      # 2. <br> -> Adds a line break.
      # 3. Value: %{{y:.2f}} -> Shows Y-value with 2 decimal places.
      # 4. {unit_abbr} -> Inserts your Python variable ('m' or 'ft').
      # 5. <extra></extra> -> Hides the trace name box on the side.
      hovertemplate=f'Date: %{{x|%Y-%m-%d}}<br>Value: %{{y:.2f}} {unit_abbr}<extra></extra>'
  ))

  # Update the layout with a dynamic y-axis title
  fig.update_layout(
      title=f'Inverse Monthly Extremes for {station} {stationName} ({levelType})',
      xaxis_title='Year',
      yaxis_title=f'Detrended Extreme Level ({unit_abbr} relative to {datum})',
      template='plotly_white'
  )

  # Show the interactive plot
fig.show()

## **Step 5**
##  Perform generalized extreme value analyses on the extreme values for all 12 months

 Then using the resulting exceedance location, scale, and shape parameters, calculate the exceedance levels for these return periods:

1.01, 2, 10, 50, and 100 years

Finally calculate the 90% confidence intervals using the normal method for the return levels below 2 years and the profile likelihood method for the return levels of 2 years or longer.

In [ ]:
%%R -i myextremes -i levelType -o gev_params -o gev_params_se -o exceed_levs -o months -o AA -o len_rp
# Load the extRemes package
library(extRemes)

# Initialize to store GEV results for each month
gev_params <- data.frame(nrow=12,ncol=3)
gev_params_se <- data.frame(nrow=12,ncol=3)
exceed_levs <- data.frame(nrow=60,ncol=6) # 12 months * 5 return periods

dimA <- dim(myextremes)
months <- c("JAN","FEB","MAR","APR","MAY","JUN","JUL","AUG","SEP","OCT","NOV","DEC")

# Fix 1: Dynamically set the number of rows
n_years <- max(table(myextremes[[1]]))
print(paste("Detected", n_years, "years of data."))

# Create the AA matrix with the correct number of rows
AA <- matrix(nrow=n_years, ncol=12)
colnames(AA) <- months

# Split the monthly extreme series into 12 series according to the month of the year
for (n in 1:12) {
  i=0
  for (k in 1:dimA[1]){

    # Fix 2 & 3: Use correct indexing
    # Column 1 is "month", Column 6 is "detrended_value"
    if (myextremes[[1]][k] == n) {
      AA[i+1,n] <- myextremes[[6]][k]
      i <- i + 1
    }
  }
}

# Iterate through the months and perform GEV analysis
for (j in 1:12) {
  # Extract the detrended values for the current month
  data_for_month <- AA[,j]
  print(data_for_month)

  # Check if we have enough non-NA data points to fit GEV (e.g., > 10)
  if (sum(!is.na(data_for_month)) > 10) {
    # Fit the GEV distribution to the monthly data
    gev_fit <- fevd(data_for_month, type = "GEV", method = "MLE", na.action=na.exclude)

    summfit <- summary(gev_fit)

    # Extract parameters
    gev_params[j,1] = summfit$par[1]
    gev_params_se[j,1] = summfit$se.theta[1]
    gev_params[j,2] = summfit$par[2]
    gev_params_se[j,2] = summfit$se.theta[2]
    gev_params[j,3] = summfit$par[3]
    gev_params_se[j,3] = summfit$se.theta[3]

    rp <- c(1.01,2,10,50,100)
    len_rp <- length(rp)
    expand_high <- 1.6
    expand_low <- 1.6
    k=1

    for (m in rp)
    {

      # First, calculate the "normal" CI as a fallback
      citemp <- ci(gev_fit, type='return.level', return.period = m, method="normal", alpha=0.1)
      middle = citemp[2]
      if ( m < 2) lower = middle - citemp[1]
      if ( m < 2) upper = citemp[3] - middle

      # If shape is ok, *try* to use the better "proflik" method
      if ( m >= 2 & gev_params[j,3] >= -0.3) {

        proflik_ci <- tryCatch(
            {
                # Try to run the proflik method
                ci(gev_fit, verbose = FALSE, type = "return.level", xrange = c(middle - lower * expand_low, middle + upper * expand_high),
                   return.period = m, method = "proflik", alpha = 0.1, nint = 30)
            },
            error = function(e) {
                # If an error (like the spline error) happens, print a warning
                print(paste("Profile likelihood failed for month", j, "RP", m, "- Using normal CI. Error:", e$message))
                # Return NULL to signify failure
                return(NULL)
            }
        )

        # 3. If proflik_ci is NOT NULL (i.e., it succeeded), overwrite citemp
        if (!is.null(proflik_ci)) {
            citemp <- proflik_ci
        }
        # If it failed, citemp just keeps the "normal" value
      }

      print(paste("exceedance level for ", m,"year return period"))
      print(paste("lower", "middle", "upper"))
      print(as.vector(round(citemp,3)))

      # Save the confidence intervals
      lower = citemp[2] - citemp[1]
      upper = citemp[3] - citemp[2]

      exceed_levs[len_rp*(j-1) + k,1] <- -999980 - j
      exceed_levs[len_rp*(j-1) + k,2] <- 1 / m
      exceed_levs[len_rp*(j-1) + k,3] <- m
      exceed_levs[len_rp*(j-1) + k,4] <- citemp[2]
      exceed_levs[len_rp*(j-1) + k,5] <- citemp[1]
      exceed_levs[len_rp*(j-1) + k,6] <- citemp[3]

      if (levelType == 'low') exceed_levs[len_rp*(j-1) + k,4] <- -exceed_levs[len_rp*(j-1) + k,4]
      if (levelType == 'low') exceed_levs[len_rp*(j-1) + k,5] <- -exceed_levs[len_rp*(j-1) + k,5]
      if (levelType == 'low') exceed_levs[len_rp*(j-1) + k,6] <- -exceed_levs[len_rp*(j-1) + k,6]

      k = k + 1
    }
  } else {
    # If no data for the month, print a message and skip
    print(paste("Skipping month", j, "(", months[j], ") due to insufficient data"))
  }
}

# The monthly_gev_results list in R will now contain the GEV fit objects
# for each month. This list will be transferred back to Python.
colnames(gev_params) <- c("exceed_location","exceed_scale","exceed_shape")
colnames(gev_params_se) <- c("exceed_location_sd","exceed_scale_sd","exceed_shape_sd")

colnames(exceed_levs) <- c("exceed_id","exceed_prob","return_period","ext_lvl","ext_lvl_minus","ext_lvl_plus")

[1] "Detected 98 years of data."
 [1] 0.05598620 0.26513626 0.62529958 0.28546289 0.46362620 0.85477626
 [7] 0.14893958 0.54010289 0.20026620 0.77441626 0.37357958 0.30774289
[13] 0.91290620 0.48105626 0.32421958 0.37454620 0.30969626 0.82285958
[19] 0.14702289 0.35518620 0.29033626 0.07149958 0.28066289 0.33582620
[25] 0.36197626 0.69213958 0.41330289 0.68246620 0.31161626 0.30677958
[31] 0.27194289 0.54110620 0.50625626 0.13541958 0.28258289 0.46074620
[37] 0.76089626 0.60405958 0.20322289 0.41138620 0.55853626 0.46269958
[43] 0.18386289 0.27002620 0.22517626 0.23633958 0.16450289 0.43366620
[49] 0.09381626 0.53697958 0.29714289 0.48430620 0.01345626 0.16661958
[55] 0.72278289 0.29994620 0.29209626 0.20825958 0.84042289 0.16858620
[61] 0.17873626 0.03089958 0.33306289 0.66422620 0.66837626 0.36153958
[67] 0.22270289 0.39486620 0.19201626 0.97317958 0.33434289 0.79050620
[73] 0.11165626 0.25581958 0.36598289 0.19814620 0.30429626 0.44445958
[79] 0.11762289 0.08778620 0.17293626 0.3080

In [ ]:
# @title Format the results

# invert exceed_location if levelType is "low"

# warning: each time this cell is run the exceed_location parameter is inverted if levelType == 'low'

# exceed_location should probably be negative if levelType == 'low'

exceed_params = pd.DataFrame()
exceed_params = pd.DataFrame(columns=["station_id","exceed_type","exceed_location","exceed_location_sd","exceed_scale","exceed_scale_sd","exceed_shape","exceed_shape_sd",
                                       "last_data_year"],index=[1,2,3,4,5,6,7,8,9])
gev_type = 'GEV_HI'
if levelType == 'low':
  for i in [1,2,3,4,5,6,7,8,9,10,11,12]:
     gev_params.iloc[i-1,0] = -gev_params.iloc[i-1,0]
  gev_type = 'GEV_LO'

for i in [1,2,3,4,5,6,7,8,9,10,11,12]:

  last_data_year = myextremes.iloc[myextremes.shape[0]-2,1]
  gevtype = gev_type +  "_" + months[i-1]
  exceed_params.loc[i] = [station, gevtype, gev_params.iloc[i-1,0], gev_params_se.iloc[i-1,0], gev_params.iloc[i-1,1], gev_params_se.iloc[i-1,1],
                          gev_params.iloc[i-1,2], gev_params_se.iloc[i-1,2], last_data_year]

exceed_params

,station_id,exceed_type,exceed_location,exceed_location_sd,exceed_scale,exceed_scale_sd,exceed_shape,exceed_shape_sd,last_data_year
1,8638610,GEV_HI_JAN,0.264291,0.019676,0.168255,0.014717,0.043177,0.089144,2025
2,8638610,GEV_HI_FEB,0.244679,0.016991,0.148886,0.01226,-0.008185,0.069803,2025
3,8638610,GEV_HI_MAR,0.27462,0.016854,0.150797,0.012074,0.026195,0.058985,2025
4,8638610,GEV_HI_APR,0.285009,0.016159,0.139626,0.012145,0.08555,0.07914,2025
5,8638610,GEV_HI_MAY,0.306025,0.015512,0.132974,0.011363,-0.029655,0.088315,2025
6,8638610,GEV_HI_JUN,0.272392,0.012077,0.104429,0.008722,-0.109334,0.081836,2025
7,8638610,GEV_HI_JUL,0.19741,0.009642,0.087345,0.006756,0.005447,0.0491,2025
8,8638610,GEV_HI_AUG,0.266426,0.015143,0.134739,0.011797,0.182878,0.071536,2025
9,8638610,GEV_HI_SEP,0.403712,0.020436,0.175029,0.016004,0.155307,0.091202,2025
10,8638610,GEV_HI_OCT,0.452886,0.018571,0.160587,0.013698,0.012477,0.084568,2025


## **Step 7**
## Visualize exceedance levels by season

### Subtask:
Create plots showing the relationship between exceedance levels and the month of the year.

In [ ]:
# Make sure you have your DataFrame 'exceed_levs' and other variables
# (station, stationName, levelType, datum, months) defined.

# Define the list of months for the x-axis ticks
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

# 1. Create a figure object
fig = go.Figure()

# Define the return periods you want to plot and their corresponding colors.
# I've added the 50-year return period (for the 2% probability level).
# You can change 'orange' to any color you like!
color_map = {
    100.0: 'red',     # 1% probability
    50.0:  'orange',  # 2% probability
    10.0:  'gold',    # 10% probability
    2.0:   'blue',    # 50% probability
    1.01:  'purple'   # 99% probability
}

# Get an ordered list of the return periods to plot
return_periods_to_plot = sorted(color_map.keys(), reverse=True)

for rp in return_periods_to_plot:
    # 1. Filter the DataFrame for the current return period.
    # The data is already in the correct month order from the R script.
    rp_data = exceed_levs[exceed_levs['return_period'] == rp]

    # 2. Add the trace to the figure
    fig.add_trace(go.Scatter(
        x=months,
        y=rp_data['ext_lvl'],
        mode='lines+markers',
        line=dict(color=color_map.get(rp)),
        marker=dict(color=color_map.get(rp)),
        name=f'{100/rp:.0f}% Probability ({int(rp)}-year RP)'
    ))

# 3. Update layout
fig.update_layout(
    title=f'Seasonal Variation of Exceedance Probability Levels for {station} {stationName}',
    yaxis_title=f'Exceedance Level (meters relative to {datum})',
    hovermode='x unified',
    title_x=0.5,
    showlegend=True,
    # These two lines were moved up to the main layout level
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis=dict(
        tickvals=list(range(12)),
        ticktext=months
        # The background color properties were removed from here
    )
)

# 4. Update traces to include custom hover information
# Using a f-string for a cleaner hover template
fig.update_traces(
    hovertemplate='Exceedance Level: %{y:.2f} meters<extra></extra>'
)

# 5. Show the figure
fig.show()

## **Takeaways**

###Thank you for taking the time to walk through this CO-OPS 12 Month Extreme Value Analysis Notebook

###Some key takeaways include:

*   How to access CO-OPS data from the CO-OPS API (here is a [URL builder](https://tidesandcurrents.noaa.gov/api-helper/url-generator.html) if you want to explore other CO-OPS data)
*   How to detrend monthly sea level data
*   How to calculate Seasonal Annual Probability of Exceedance (AEP) from monthly high and low water level data